In [0]:
accounts_df = spark.table("bronze_accounts")
transactions_df = spark.table("bronze_transactions")
fraud_df = spark.table("bronze_frauds")

In [0]:
accounts_df.show(5)
accounts_df.printSchema()
accounts_df.count()


+----------+---------------+------------+------------+----------------+----------+------------+
|account_id|  customer_name|account_type|opening_date|          branch|kyc_status|credit_limit|
+----------+---------------+------------+------------+----------------+----------+------------+
| ACC-00001|  Rajesh Kapoor|     savings|  2020-09-21|     Mumbai_Main|   pending|     1000000|
| ACC-00002|Sunita Deshmukh|         NRI|  2021-10-14|        Delhi_CP|   pending|     1000000|
| ACC-00003|     Anil Reddy|     current|  2022-07-03|    Bangalore_MG|  verified|        NULL|
| ACC-00004|     Meena Iyer|         NRI|  2021-10-26|    Bangalore_MG|  verified|      100000|
| ACC-00005|   Suresh Patel|     current|  2016-12-23|Hyderabad_Hitech|   expired|       50000|
+----------+---------------+------------+------------+----------------+----------+------------+
only showing top 5 rows
root
 |-- account_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- account_type: s

50

In [0]:

accounts_df.dropDuplicates().count()

50

In [0]:
from pyspark.sql.functions import col, sum, when
accounts_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in accounts_df.columns
]).show()

+----------+-------------+------------+------------+------+----------+------------+
|account_id|customer_name|account_type|opening_date|branch|kyc_status|credit_limit|
+----------+-------------+------------+------------+------+----------+------------+
|         0|            0|           0|           0|     0|         0|           9|
+----------+-------------+------------+------------+------+----------+------------+



In [0]:
from pyspark.sql.functions import avg
avg_credit_limit=accounts_df.select(avg("credit_limit")).first()[0]
accounts_clean = accounts_df.fillna({"credit_limit": int(avg_credit_limit)})

In [0]:
accounts_clean.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in accounts_clean.columns
]).show()

+----------+-------------+------------+------------+------+----------+------------+
|account_id|customer_name|account_type|opening_date|branch|kyc_status|credit_limit|
+----------+-------------+------------+------------+------+----------+------------+
|         0|            0|           0|           0|     0|         0|           0|
+----------+-------------+------------+------------+------+----------+------------+



In [0]:
accounts_clean.write.format("delta").mode("overwrite").saveAsTable("silver_accounts")

In [0]:
spark.sql("SHOW TABLES").show()

+--------+-------------------+-----------+
|database|          tableName|isTemporary|
+--------+-------------------+-----------+
| default|    bronze_accounts|      false|
| default|      bronze_frauds|      false|
| default|bronze_transactions|      false|
| default|   combined_dataset|      false|
| default|  sample_superstore|      false|
| default|    silver_accounts|      false|
+--------+-------------------+-----------+



In [0]:
transactions_df.count()
transactions_df.dropDuplicates().count()
transactions_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in transactions_df.columns
]).show()

+------+----------+--------+--------+------+--------+----+----------------+
|txn_id|account_id|txn_date|txn_type|amount|merchant|city|is_international|
+------+----------+--------+--------+------+--------+----+----------------+
|     0|         0|       0|       0|     0|       7|   0|               0|
+------+----------+--------+--------+------+--------+----+----------------+



In [0]:
transactions_clean = transactions_df.fillna({
    "merchant": "Unknown"
})

In [0]:
transactions_clean.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in transactions_clean.columns
]).show()

+------+----------+--------+--------+------+--------+----+----------------+
|txn_id|account_id|txn_date|txn_type|amount|merchant|city|is_international|
+------+----------+--------+--------+------+--------+----+----------------+
|     0|         0|       0|       0|     0|       0|   0|               0|
+------+----------+--------+--------+------+--------+----+----------------+



In [0]:
transactions_clean.write.format("delta").mode("overwrite").saveAsTable("silver_transactions")

In [0]:

fraud_df.printSchema()
fraud_df.show(5)
fraud_df.count()

root
 |-- account_id: string (nullable = true)
 |-- fraud_type: string (nullable = true)
 |-- flagged_date: date (nullable = true)

+----------+----------------+------------+
|account_id|      fraud_type|flagged_date|
+----------+----------------+------------+
| ACC-00021|    card_cloning|  2026-01-14|
| ACC-00029|money_laundering|  2026-01-13|
| ACC-00031|        phishing|  2026-03-08|
| ACC-00009|  identity_theft|  2026-01-18|
| ACC-00017|account_takeover|  2026-03-08|
+----------+----------------+------------+
only showing top 5 rows


10

In [0]:
fraud_df.dropDuplicates().count()

10

In [0]:
fraud_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in fraud_df.columns
]).show()

+----------+----------+------------+
|account_id|fraud_type|flagged_date|
+----------+----------+------------+
|         0|         0|           0|
+----------+----------+------------+



In [0]:
fraud_df.write.format("delta").mode("overwrite").saveAsTable("silver_frauds")

In [0]:
silver_combined = (
    transactions_clean.alias("t")
    .join(
        accounts_clean.alias("a"),
        on="account_id",
        how="left"
    )
    .join(
        fraud_df.alias("f"),
        on="account_id",
        how="left"
    )
)

In [0]:
display(silver_combined)
silver_combined.count()
silver_combined.printSchema()

account_id,txn_id,txn_date,txn_type,amount,merchant,city,is_international,customer_name,account_type,opening_date,branch,kyc_status,credit_limit,fraud_type,flagged_date
ACC-00056,TXN-000001,2026-01-09,debit,915931.22,Unknown,Singapore,yes,null,null,null,null,null,null,account_takeover,2026-02-20
ACC-00025,TXN-000002,2026-01-11,transfer,23892.36,BookMyShow,Bangalore,no,Hemant Jain,NRI,2023-02-17,Hyderabad_Hitech,verified,500000,null,null
ACC-00029,TXN-000003,2026-02-03,credit,15277.03,Swiggy,New York,yes,Tanuja Nair,salary,2024-05-02,Pune_FC,verified,100000,money_laundering,2026-01-13
ACC-00007,TXN-000004,2026-03-05,withdrawal,5498.74,ATM,Delhi,no,Prakash Rao,NRI,2016-10-04,Pune_FC,expired,500000,null,null
ACC-00029,TXN-000005,2026-02-23,debit,919.78,Flipkart,Chennai,no,Tanuja Nair,salary,2024-05-02,Pune_FC,verified,100000,money_laundering,2026-01-13
ACC-00047,TXN-000006,2026-02-09,debit,22417.98,Swiggy,Mumbai,no,Pawan Arora,savings,2024-02-02,Hyderabad_Hitech,pending,100000,null,null
ACC-00003,TXN-000007,2026-02-22,transfer,3832.61,BigBasket,Singapore,yes,Anil Reddy,current,2022-07-03,Bangalore_MG,verified,342682,null,null
ACC-00037,TXN-000008,2026-03-01,withdrawal,11979.58,PhonePe,Delhi,no,Naveen Chadha,salary,2015-11-09,Kolkata_Park,expired,50000,null,null
ACC-00035,TXN-000009,2026-01-08,withdrawal,23856.18,Unknown_Merchant,New York,yes,Tarun Bose,savings,2017-10-27,Delhi_CP,expired,500000,null,null
ACC-00008,TXN-000010,2026-03-19,transfer,8369.32,BookMyShow,Singapore,yes,Lata Mishra,salary,2022-11-22,Mumbai_Main,verified,100000,null,null


root
 |-- account_id: string (nullable = true)
 |-- txn_id: string (nullable = true)
 |-- txn_date: date (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- merchant: string (nullable = false)
 |-- city: string (nullable = true)
 |-- is_international: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- opening_date: date (nullable = true)
 |-- branch: string (nullable = true)
 |-- kyc_status: string (nullable = true)
 |-- credit_limit: integer (nullable = true)
 |-- fraud_type: string (nullable = true)
 |-- flagged_date: date (nullable = true)



In [0]:
silver_combined.write.format("delta").mode("overwrite").saveAsTable("silver_combined_dataset")